In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from torch_geometric.nn import HeteroConv, GATConv, HGTConv
from torch_geometric.nn import GCNConv, GATConv, GINConv
from torch_scatter import scatter

import pickle
import pandas as pd
import numpy as np

# Add parent directory to path for imports
import os
import sys
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
sys.path.append(os.path.dirname(base_dir))
from utils.helper import networkx_to_hetero_data, get_device, set_random_seeds
from typing import Any, Dict, Tuple, List

/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


load graph and convert to HeteroData

In [2]:
patient_kg_path = "./data/KG/patient_kg.pkl"

with open(patient_kg_path, 'rb') as f:
    G = pickle.load(f) # patient_kg

# prepare HeteroData
data, new_node_mappings = networkx_to_hetero_data(G)

Found 16 node types.
Add y label and train, val, test mask to data['Patient].
Found 996 edge types.
Add (patient,rel,protein) edge_weights to HeteroData
Conversion complete!


In [3]:
data['Protein'].num_nodes

1301

## Model design

In [102]:
class RelevancePropagationLayer(nn.Module):
    """
    Propagates disease-relevance scores through edges.
    Scores flow from neighbors to central node.
    Different edge types have different learnable aggregation weights.
    """
    def __init__(self, edge_types, aggregation='mean') -> None:
        """_summary_

        Args:
            edge_types (list): list of edge type tuples (src, rel, dst)
            aggregation (str, optional): Defaults to 'mean'. Can be 'mean', 'sum'
        """
        super().__init__()
        
        self.edge_types = edge_types
        self.aggr = aggregation

        # Initialize learnable aggregation weight per edge type
        self.edge_type_weight = nn.ParameterDict({
            "__".join(edge_type): nn.Parameter(torch.ones(1))
            for edge_type in edge_types
        })
        # if necessary? add a gate to control the updaing of new relevance scores
        self.gate = nn.Sequential(
            nn.Linear(1,16),
            nn.ReLU(),
            nn.Linear(16,1),
            nn.Sigmoid() # -> [0,1]
        )
    def forward(self, relevance_dict, edge_index_dict):
        """Aggregate and update relevance scores.

        Args:
            relevance_dict (dict): {node_type: [num_nodes, 1]}
            edge_index_dict (dict): {edge_type: [2, num_edges]}
        Returns:
            new_relevance_dict (dict): {node_type: [num_nodes, 1]}
        """
        # initialize aggregated scores dict
        aggregated_dict = {
            node_type: torch.zeros_like(scores)
            for node_type, scores in relevance_dict.items()
        }
        # count incoming messages per node to average
        message_counts_dict = {
            node_type: torch.zeros(scores.size(0))
            for node_type, scores in relevance_dict.items()
        }
        # aggregate relevance from each edge
        for edge_type, edge_index in edge_index_dict.items():
            src_type, _, dst_type = edge_type
            src_idx, dst_idx = edge_index[0], edge_index[1]

            # old node_relevance scores
            src_scores = relevance_dict[src_type]
            # edge_type weight
            edge_type_key = "__".join(edge_type)
            edge_weight = torch.sigmoid(self.edge_type_weight[edge_type_key])
            # aggregating 
            messages = src_scores[src_idx] * edge_weight # [num_edges, 1]
            aggregated_dict[dst_type] = aggregated_dict[dst_type].scatter_add(
                0, dst_idx.unsqueeze(1).expand_as(messages), messages
            )
            # count messages to dst_node
            message_counts_dict[dst_type] = message_counts_dict[dst_type].scatter_add(
                0, dst_idx, torch.ones(dst_idx.size(0))
            )
        # average aggregation
        if self.aggr == 'mean':
            for node_type in aggregated_dict.keys():
                counts = message_counts_dict[node_type] #[num_nodes]
                # Need to be careful about dimension match in division.
                aggregated_dict[node_type] = aggregated_dict[node_type]/counts.view(-1, 1).clamp(min=1)
        # Update new relevance scores based on aggegated scores and old scores
        new_relevance_dict = {}
        for node_type in relevance_dict.keys():
            old = relevance_dict[node_type]
            aggregated = aggregated_dict[node_type]
            # gate control
            new_relevance_dict[node_type] = self.gate(old) * aggregated + (1-self.gate(old))*old
        
        return new_relevance_dict

In [109]:
class RelevanceMessagePassing(MessagePassing):
    """
    Message passing layer with edge coefficients.
    attention_rule = edge_coefficient = f(src_relevance, dst_relevance, edge_weight).
    Combines with attention_neural.
    """
    def __init__(self, in_channels, out_channels, heads) -> None:
        super().__init__(aggr='add', node_dim=0)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.heads = heads

        # GAT part
        if isinstance(in_channels, tuple):
            in_channels_src, in_channels_dst = in_channels
        else:
            in_channels_src=in_channels_dst = in_channels
        # project features to attention space [num_edges, heads, out_channels]
        self.lin_src = nn.Linear(in_channels_src, heads*out_channels)
        self.lin_dst = nn.Linear(in_channels_dst, heads*out_channels)

        # attention coefficient vector [1, heads, 2*out_channels]
        self.att_neural = nn.Parameter(torch.Tensor(1, heads, 2*out_channels))
        nn.init.xavier_uniform_(self.att_neural)

        # edge_coefficient: input is [src_relevance, dst_relevance, edge_weight]
        self.relevance_coeff = nn.Sequential(
            nn.Linear(3,16),
            nn.ReLU(),
            nn.Linear(16, heads),
            nn.Sigmoid() # -> [0,1]
        )
    def forward(self, x, edge_index, relevance_scores, edge_weight=None):
        """Passing neural message.

        Args:
            x: Node features (tuple or tensor)
            edge_index: [2, num_edges]
            relevance_scores: (src_scores, dst_scores)
            edge_weight: [num_edges] optional edge weights.
        
        Returns:
            out (Tensor):
        """
        if isinstance(x, tuple):
            x_src, x_dst = x[0], x[1]
        else:
            x_src = x_dst = x
        # project node features to sttention head space
        x_src = self.lin_src(x_src).view(-1, self.heads, self.out_channels)
        x_dst = self.lin_dst(x_dst).view(-1, self.heads, self.out_channels)

        # neural message aggregation
        out = self.propagate(
            edge_index,
            x=(x_src, x_dst),
            relevance_scores = relevance_scores,
            edge_weight = edge_weight
        )
        return out.mean(dim=1) # to merge attention heads
    
    def message(self, x_j, x_i, relevance_scores, edge_weight, index, ptr, size_i):
        """Neural and Relevance message aggregation per edge_type.

        Args:
            x_j (_type_): x_src
            x_i (_type_): x_dst
            relevance_scores (tuple): ([num_edges, 1], [num_edges, 1])
            edge_weight (Tensor): [num_edges, 1]
            index (_type_): _description_
            ptr (_type_): _description_
            size_i (_type_): _description_

        Returns:
            Tensor: [num_dst, feature_dim]
        """

        # neural attention -> [num_edges, heads]
        # alpha_neural = e_ij = LeakyReLU(a^T[ Wh_i​∣∣ Wh_j ​])
        alpha_neural = (self.att_neural * torch.cat([x_i, x_j], dim=-1)).sum(dim=-1)
        
        # disease_relevance attention
        src_scores, dst_scores = relevance_scores
        if edge_weight is None:
            edge_weight_expanded = torch.ones(src_scores.size(0), 1)
        else:
            edge_weight_expanded = edge_weight.view(-1, 1)
        
        print('message passing: relevance and edge weight size')
        print(src_scores.size())
        print(dst_scores.size())
        print(edge_weight_expanded.size())
        relevance_input = torch.cat([
            src_scores.view(-1,1), # should be [num_edges, 1]
            dst_scores.view(-1,1), # should be [num_edges, 1]
            edge_weight_expanded # [num_edges, 1]
        ], dim=1)
        alpha_relevance = self.relevance_coeff(relevance_input.float())

        # Combine both attentions
        alpha_combined = alpha_neural + alpha_relevance
        alpha = softmax(alpha_combined, index, ptr, size_i)

        return x_j * alpha.unsqueeze(-1)   

In [110]:
class RMPGAT(nn.Module):
    """
    Relevance Propagation GAT with relevance attention to control message flow.
    """
    def __init__(self, data, in_channels, hidden_channels, out_channels,
                 num_layers=3, heads = 2, dropout_rate=0.5):
        super().__init__()

        self.num_layers = num_layers
        self.dropout_rate = dropout_rate
        self.node_types = data.node_types
        self.edge_types = data.edge_types

        # 1. Neural layers
        self.embeddings = nn.ModuleDict({
            node_type: nn.Embedding(num_nodes, in_channels)
            for node_type, num_nodes in {nt: data[nt].num_nodes for nt in data.node_types}.items()
        })
        self.convs = nn.ModuleList()
        for i in range(num_layers):
            layer_dict = nn.ModuleDict()
            input_dim = in_channels if i == 0 else hidden_channels

            for edge_type in self.edge_types:
                layer_dict['__'.join(edge_type)] = RelevanceMessagePassing(
                    in_channels=input_dim,
                    out_channels=hidden_channels,
                    heads=heads
                )
            self.convs.append(layer_dict)
       
        # 2. Relevance propagation layers
        self.relevance_layers = nn.ModuleList([
            RelevancePropagationLayer(edge_types=self.edge_types, aggregation='mean')
            for _ in range(num_layers)
        ])
        self.relevance_params = nn.ParameterDict() # relevance scores are learnable parameters now

        # 3. classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels //2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_channels //2, out_channels)
        )

        # 4. Link Prediction Head 
        self.rel_weights = nn.ParameterDict({
            "__".join(edge_type): nn.Parameter(torch.ones(hidden_channels))
            for edge_type in self.edge_types
        })
        for param in self.rel_weights.values():
            torch.nn.init.xavier_uniform_(param.unsqueeze(0))
    
    def initialize_relevances(self, initial_relevance_dict, data):
        """
        Initialize node_relevances as nn.Parameters.
        The initial relevance for patients are set to 0.
        """
        for node_type in self.node_types:
            if node_type in initial_relevance_dict:
                init_score = initial_relevance_dict[node_type].clone()
            else:
                num_nodes = data[node_type].num_nodes
                init_score = torch.zeros(num_nodes)
            # {node_type: [num_nodes, 1]}
            self.relevance_params[node_type] = nn.Parameter(init_score.unsqueeze(1))
    
    def apply_one_layer(self, x_dict, edge_index_dict, relevance_dict,
                        edge_weight_dict, layer_idx):
        """Mannually looping over edge type MessagePassingLayers to avoid using HeteroConv."""
        out_dict = {node_type: [] for node_type in x_dict.keys()}
        conv_layer = self.convs[layer_idx]

        for edge_type, edge_index in edge_index_dict.items():
            src_type, _, dst_type = edge_type
            edge_type_key = '__'.join(edge_type)
            conv = conv_layer[edge_type_key]

            x_src= x_dict[src_type]
            x_dst = x_dict[dst_type]
            src_idx, dst_idx = edge_index[0], edge_index[1]
            relevance_scores = (
                relevance_dict[src_type][src_idx],
                relevance_dict[dst_type][dst_idx]
            )
            print('apply one layer: checking dimensions')
            print('edge_type: ',edge_type)
            print('src id: ', src_idx)
            print('dst_id: ', dst_idx)
            print('relevance score dimension: \n', 
                  relevance_dict[src_type][src_idx].size(), relevance_dict[dst_type][dst_idx].size())

            # edge_weight = expression value or None (for KG edges)
            edge_weight = edge_weight_dict.get(edge_type, None)
            
            out = conv(
                x=(x_src, x_dst),
                edge_index=edge_index,
                relevance_scores= relevance_scores,
                edge_weight=edge_weight
            )
            out_dict[dst_type].append(out)
        
        for node_type in out_dict.keys():
            if len(out_dict[node_type]) > 0:
                out_dict[node_type] = torch.stack(out_dict[node_type]).mean(dim=0)
            else:
                out_dict[node_type] = x_dict[node_type]
        return out_dict
    
    def forward(self, edge_index_dict, initial_relevance_dict, edge_weight_dict):

        x_dict = {node_type: emb.weight for node_type, emb in self.embeddings.items()}
        # initialise relevance scores
        # self.initialize_relevances(initial_relevance_dict, data)
        relevance_dict = {
            node_type: params.clone()
            for node_type, params in self.relevance_params.items()
        }
        relevance_history = [{k:v.clone() for k,v in relevance_dict.items()}]

        for layer_idx in range(self.num_layers):
            print('GNN layer: ', layer_idx)
            x_dict = self.apply_one_layer(
                x_dict, edge_index_dict, relevance_dict, edge_weight_dict, layer_idx
            )
            x_dict = {k: F.elu(v) for k, v in x_dict.items()}
            x_dict = {k: F.dropout(v, p=self.dropout_rate, training=self.training) 
                      for k, v in x_dict.items()}
            
            # relevance score propagation
            relevance_dict = self.relevance_layers[layer_idx](
                relevance_dict, edge_index_dict
            )
            relevance_history.append({k:v.clone() for k,v in relevance_dict.items()})
        
        # classifier
        h_patient = x_dict['Patient']
        logits = self.classifier(h_patient)
        log_probs = F.log_softmax(logits, dim=1)

        return x_dict, log_probs, relevance_history
    
    def decode(self, h_dict, edge_index, edge_type):
        """Link Prediction Decoder: Predicts probability of edges"""
        u_type, rel, v_type = edge_type
        src, dst = edge_index
        
        h_src = h_dict[u_type][src]
        h_dst = h_dict[v_type][dst]
        
        rel_key = "__".join(edge_type)
        rel_w = self.rel_weights[rel_key]
        return (h_src * rel_w * h_dst).sum(dim=-1)
    
    def get_edgetype_propagation_weight(self):
        weights = {}
        for layer_idx, layer in enumerate(self.relevance_layers):
            weights[f'layer_{layer_idx}'] = {
                edge_type: torch.sigmoid(weight).item()
                for edge_type, weight in layer.edge_type_weight.items()
            }
        return weights

In [111]:
def train_epoch(model, data, edge_index_dict, initial_relevance_dict, edge_weight_dict, optimizer):

    model.train()
    optimizer.zero_grad()

    # 1. forward pass
    h_dict, log_probs, relevance_history = model(edge_index_dict, initial_relevance_dict, edge_weight_dict)

    # classification loss
    data['Patient'].y = torch.as_tensor(data['Patient'].y).long().squeeze()
    print(f"Patient Mask Shape: {data['Patient'].train_mask.shape}")
    print(f"Log Probs Shape: {log_probs.shape}")
    print(f"Target Y Shape: {data['Patient'].y.shape}")
    loss_cls = F.nll_loss(log_probs[data['Patient'].train_mask], data['Patient'].y[data['Patient'].train_mask])

    # link prediction loss
    loss_lp = 0
    for edge_type in edge_index_dict.keys():
        pos_edge_index = edge_index_dict[edge_type]
        
        # Positive Scores
        pos_scores = model.decode(h_dict, pos_edge_index, edge_type)
        
        # Negative Sampling: Randomly shuffle destination nodes
        neg_edge_index = pos_edge_index.clone()
        num_nodes_v = data[edge_type[2]].num_nodes
        neg_edge_index[1] = torch.randint(0, num_nodes_v, (pos_edge_index.size(1),))
        
        #neg_edge_index[1] = neg_edge_index[1][torch.randperm(neg_edge_index.size(1))]
        neg_scores = model.decode(h_dict, neg_edge_index, edge_type)
        
        # Ranking Loss: Encourage pos_scores > neg_scores
        loss_lp += -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-15).mean()
    
    loss_lp = loss_lp / len(data.edge_types) # to prevent link prediction loss becomes too huge
    total_loss = loss_cls + loss_lp
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    result = {
        'total loss': total_loss,
        'classification loss': loss_cls,
        'LP loss': loss_lp,
        'relevance history': relevance_history
    }

    return result

test model training

In [112]:
model = RMPGAT(
    data=data,
    in_channels=32,
    hidden_channels=64,
    out_channels=2,
    num_layers=3,
    heads=2,
    dropout_rate=0.5
)
model

RMPGAT(
  (embeddings): ModuleDict(
    (Gene): Embedding(137, 32)
    (Abundance): Embedding(408, 32)
    (BiologicalProcess): Embedding(446, 32)
    (Activity): Embedding(546, 32)
    (Pathology): Embedding(126, 32)
    (MicroRna): Embedding(46, 32)
    (Protein): Embedding(1301, 32)
    (Rna): Embedding(111, 32)
    (Translocation): Embedding(64, 32)
    (Reaction): Embedding(13, 32)
    (Degradation): Embedding(56, 32)
    (CellSecretion): Embedding(30, 32)
    (CellSurfaceExpression): Embedding(3, 32)
    (Complex): Embedding(373, 32)
    (Composite): Embedding(72, 32)
    (Patient): Embedding(744, 32)
  )
  (convs): ModuleList(
    (0): ModuleDict(
      (Gene__decreases__Protein): RelevanceMessagePassing(32, 64)
      (Gene__increases__Gene): RelevanceMessagePassing(32, 64)
      (Gene__increases__Protein): RelevanceMessagePassing(32, 64)
      (Gene__rev_association__Gene): RelevanceMessagePassing(32, 64)
      (Gene__rev_association__Protein): RelevanceMessagePassing(32, 64)
 

In [107]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 0.005,
    weight_decay=5e-4
)

edge_index_dict = {edge_type: data[edge_type].edge_index for edge_type in data.edge_types}
initial_relevance_dict = {node_type: data[node_type].relevance for node_type in data.node_types if node_type != 'Patient'}
edge_weight_dict = {}
for edge_type in data.edge_types:
    if 'Patient' in edge_type:
        ew = data[edge_type].edge_weight
    else:
        ew = None
    
    edge_weight_dict[edge_type] = ew

In [ ]:
initial_relevance_dict["('Gene', 'decreases', 'Protein')"]

In [113]:
# initialize relevance_params
model.initialize_relevances(initial_relevance_dict, data)
result_history = []
for epoch in range(2):
    epoch_result = train_epoch(
        model=model,
        data=data,
        edge_index_dict=edge_index_dict,
        initial_relevance_dict=initial_relevance_dict,
        edge_weight_dict=edge_weight_dict,
        optimizer=optimizer
    )
    result_history.append(epoch_result)

GNN layer:  0
apply one layer: checking dimensions
edge_type:  ('Gene', 'decreases', 'Protein')
src id:  tensor([ 0,  6, 84, 86, 99])
dst_id:  tensor([269, 269, 269, 269, 261])
relevance score dimension: 
 torch.Size([5, 1]) torch.Size([5, 1])
message passing: relevance and edge weight size
torch.Size([5, 1])
torch.Size([5, 1])
torch.Size([5, 1])
apply one layer: checking dimensions
edge_type:  ('Gene', 'increases', 'Gene')
src id:  tensor([0])
dst_id:  tensor([84])
relevance score dimension: 
 torch.Size([1, 1]) torch.Size([1, 1])
message passing: relevance and edge weight size
torch.Size([1, 1])
torch.Size([1, 1])
torch.Size([1, 1])
apply one layer: checking dimensions
edge_type:  ('Gene', 'increases', 'Protein')
src id:  tensor([  0,   5,   9,  11,  12,  25,  28,  35,  36,  40,  83,  99, 101, 103,
        106, 107, 108, 109, 110, 111, 113, 114, 115, 116, 117, 118])
dst_id:  tensor([ 157,   80,  180,  246, 1205,  899,  261,  269,  269,  269,  269, 1170,
        1171, 1172,  288, 1175